# 완전자본잠식 12·24개월 조기예측
실제 자료만 분석합니다. 합성 단위 테스트 결과는 연구 성능으로 사용하지 않습니다.
공시일 검증·과거 상장 모집단이 없으면 실행 결과는 blocked입니다.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
root=Path.cwd()
if root.name=='notebooks': root=root.parent
if not (root/'src/capitalrisk').exists(): raise RuntimeError('Run from research/')
sys.path.insert(0,str(root/'src'))
from capitalrisk.pipeline import run
config=json.loads((root/'config.json').read_text())
config

Out[1]: 
{'seed': 20261028,
 'basis': 'OFS',
 'start_year': 2015,
 'end_year': 2025,
 'horizons': [12, 24],
 'train_end': '2018-12-31',
 'validation_start': '2021-01-01',
 'validation_end': '2021-12-31',
 'test_start': '2024-01-01',
 'test_end': '2025-12-31',
 'bootstrap_repetitions': 1000,
 'minimum_event_companies': 20,
 'sequence_quarters': 8,
 'max_statement_age_days': 185}


## 수집 상태
KIND 현재 목록은 과거 전체 모집단이 아닙니다. 원본 자료는 공개 재배포 검토 전 Git에서 제외합니다.

In [2]:
audit=root/'outputs/collection_audit.csv'
pd.read_csv(audit) if audit.exists() else '수집 기록 없음'

Out[2]: 
                            time_utc  ...                                           metadata
0   2026-10-27T18:34:40.746393+00:00  ...  {"rows": 2758, "scope": "current page only; no...
1   2026-10-27T18:34:46.241053+00:00  ...  {"rows": 2, "scope": "current page only; not h...
2   2026-10-27T18:36:41.300542+00:00  ...  {"bytes": 4694, "sha256": "cdf534640d956f068d7...
3   2026-10-27T18:36:47.591061+00:00  ...  {"bytes": 1243824, "sha256": "505a822145691308...
4   2026-10-27T18:36:52.986164+00:00  ...  {"bytes": 84475, "sha256": "176406c126425327c1...
5   2026-10-27T18:36:58.501597+00:00  ...  {"bytes": 15165, "sha256": "b15d927c2ed856783f...
6   2026-10-27T18:36:58.501895+00:00  ...                                                 {}
7   2026-10-27T18:37:32.540293+00:00  ...                                                 {}
8   2026-10-27T18:39:39.110576+00:00  ...                                     {"rows": 2758}
9   2026-10-27T18:39:44.927774+00:00  ...                    

## 실제 파이프라인 실행
검증된 입력이 없는 경우 오류 원인과 미완료 상태를 기록합니다. 정확도·ROC·SHAP를 임의 생성하지 않습니다.

In [3]:
status=run(root/'data/private/financials.csv',root/'data/private/universe.csv',config)
status

Out[3]: 
{'status': 'blocked',
 'actual_financial_rows': 0,
 'incident_companies': 0,
 'models_evaluated': 0,
 'config_sha256': '3337f262af62a353cf0761bdf728bec1767e0b7a874f7c9acf6c9ef5377145eb',
 'python': '3.12.14',
 'limitations': ["FileNotFoundError: [Errno 2] No such file or directory: '/workspace/scratch/47e82d6e6967/invest_guide/research/data/private/universe.csv'"],
 'scope': 'verified historical universe; no resampling of validation/test',
 'run_directory': 'run_20261027T184429512551Z'}


In [4]:
run_dir=root/'outputs'/status['run_directory']
metrics=sorted(run_dir.glob('metrics_*.csv'))
if metrics:
    display(pd.concat([pd.read_csv(p) for p in metrics],ignore_index=True))
else:
    print('실제 예측 성능: 평가 불가 — 검증된 입력 또는 사건 수 부족')

실제 예측 성능: 평가 불가 — 검증된 입력 또는 사건 수 부족


In [5]:
from IPython.display import Image, display
figures=sorted((run_dir/'figures').glob('*.png'))
for p in figures: display(Image(filename=str(p)))
if not figures: print('검증된 예측 데이터가 없어 모델 성능 그림을 생성하지 않았습니다.')

검증된 예측 데이터가 없어 모델 성능 그림을 생성하지 않았습니다.


## 해석
README와 docs/METHODOLOGY.md, outputs/REPORT.md를 함께 확인합니다. 2013 논문의 계수·원표본 재현은 미완료이며, Logistic_2013_approach는 초록의 변수 접근만 재구현한 모델입니다.